# PRW person search: final system comparison

This notebook evaluates saved checkpoints only. It does not train models or select thresholds on test. Run the cells in order after configuring the artifact manifest. Every reported value is recomputed, with checkpoint provenance saved beside the results. ResNet uses run jw7ysc43 / prw-best-model:v6 with SupCon. Reference scores are not substituted for recomputed results. Its crop re-encoding path is used as in the historical baseline; this is not labeled single-pass because SupCon alone does not establish crop-to-scene training.

| System | Gallery ReID path | ReID training constraints |
|---|---|---|
| ResNet baseline | Original-pixel crop re-encoding, shared backbone | SupCon; verify saved training configuration |
| Swin single-pass | Frozen scene features → learned RoI head | Crop-to-scene InfoNCE; only ReID head trained |
| Swin two-pass | Original-pixel crops → independent Swin | Crop-to-crop InfoNCE; encoder fine-tuned |
| Swin + DINO two-pass | Original-pixel crops → DINO CLS + upper/lower features | Crop-to-crop InfoNCE; encoder fine-tuned |

These compare complete systems, not an isolated pass-count change. DINO had a longer training budget and more tuning than the 50-epoch Swin runs. Report these differences alongside accuracy. The single-pass name refers to the gallery backbone, not query encoding.


In [ ]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("github_token")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py", "offline_mining.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


In [ ]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



In [ ]:
# Kaggle usually includes these; install only missing report dependencies.
import importlib.util
import subprocess
import sys
missing=[package for module,package in [('sklearn','scikit-learn'),('pandas','pandas'),('matplotlib','matplotlib')]
         if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
from pathlib import Path
import json
import gc
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from final_comparison import (resolve_checkpoint, load_system, fixed_occurrences,
                              occurrence_embeddings, plot_embeddings, plot_similarities,
                              plot_matched_query)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table

PREFIX='models-alma-mater-studio/person-search/'
# Exact versions supplied for Swin; latest.pt is NOT necessarily validation-best.
SYSTEMS=[
    dict(name='ResNet baseline (crop)',kind='resnet_crop',enabled=True,
         artifact=PREFIX+'prw-best-model:v6',run_id='jw7ysc43',filename='best.pt',
         selection='best supplied by user',expected_metric_loss='supcon',crop_size=(128,64),
         reference_mAP=.3170,reference_top1=.6874,
         training_note='SupCon; shared crop re-encoding baseline. Scene-path training not established'),
    dict(name='Swin single-pass',kind='swin_scene',enabled=True,
         artifact=PREFIX+'swin-reid-crop_to_scene-latest:v4',run_id='vdrppx5p',
         filename='latest.pt',selection='latest supplied by user',training_note='Frozen scene backbone; trained RoI head'),
    dict(name='Swin two-pass',kind='swin_crop',enabled=True,
         artifact=PREFIX+'swin-reid-crop_to_crop-latest:v4',run_id='tv1tvl2u',
         filename='latest.pt',selection='latest supplied by user',training_note='Separate Swin crop encoder'),
    # Defaults taken from the previously verified combined-evaluation notebook.
    dict(name='Swin + DINO',kind='dino',enabled=True,
         artifact=PREFIX+'prw-dinov3-last4-p16k4-upper-lower-infonce-256x128-valmap-best-model:v23',
         run_id='xhdye71i',filename='best.pt',selection='validation best',
         training_note='CLS + upper/lower, four trainable blocks; longer training budget'),
]
DINO_DETECTOR=dict(artifact=PREFIX+'swin-detector-512x768-best:v0',run_id='77d9acgi',filename='best.pt')
SCORE_THRESHOLD=.5  # Same fixed operating point for every system; choose on validation.
IOU_THRESHOLD=.5
NUM_VISUAL_IDS=20
OCCURRENCES_PER_ID=8
NUM_EXAMPLE_QUERIES=10
SEED=0
LOG_TO_WANDB=False  # Set True in the last cell to upload comparisons to a NEW run.
OUTPUT=Path('/kaggle/working/final_prw_comparison')
OUTPUT.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
active=[dict(s) for s in SYSTEMS if s['enabled']]
if not active:raise ValueError('Enable at least one system')
for s in SYSTEMS:
    if not s['enabled']:print('NOT EVALUATED:',s['name'],'— checkpoint/provenance still required')
for s in active:
    if not s.get('artifact') and not s.get('local_path'):raise ValueError(f"Missing checkpoint: {s['name']}")
    s['checkpoint_path']=str(resolve_checkpoint(s,OUTPUT/'checkpoints'))
detector_path=resolve_checkpoint(DINO_DETECTOR,OUTPUT/'checkpoints') if any(s['kind']=='dino' for s in active) else None
display(pd.DataFrame(active)[['name','kind','selection','artifact','training_note']])


## Matched evaluation and diagnostic samples
All metrics use every available official test query and all test frames; examples do not limit evaluation. GT-gallery results use annotation boxes and each model's native gallery feature path. Unknown-ID people remain gallery distractors.

The 20-ID visualization sample uses eight different frames per identity, chosen deterministically before loading any model. It is a diagnostic subset, not the population used to calculate mAP. Gallery t-SNE projections are fit separately with the same settings; their axes/cluster spacing are not directly comparable between panels. Similarity matrices compare the crop-query path with each system's gallery path on the same GT occurrences.


In [ ]:
sample=fixed_occurrences(PRW_ROOT,NUM_VISUAL_IDS,OCCURRENCES_PER_ID,SEED)
(OUTPUT/'visualization_sample.json').write_text(json.dumps(sample,indent=2))
print('Visualization identity IDs:',sorted({r['identity'] for r in sample}))
rows=[];provenance={};banks={};retrieval_records={}
shared_detector=None
for index,spec in enumerate(active):
    print('Evaluating:',spec['name'])
    model,info=load_system(spec,detector_path,DEVICE)
    if not spec['kind'].startswith('resnet'):
        signature=(info['detector_sha256'],info['detector_image_size'],
                   json.dumps(info['detector_config'],sort_keys=True))
        if shared_detector is None:shared_detector=signature
        elif signature!=shared_detector:raise RuntimeError('Detector weights/resolution differ: comparison stopped')
    provenance[spec['name']]=info
    for mode in ('detected','gt'):
        started=time.perf_counter()
        result=evaluate_prw(model,PRW_ROOT,DEVICE,score_threshold=SCORE_THRESHOLD,
                            iou_threshold=IOU_THRESHOLD,detector_image_size=info['detector_image_size'],
                            gallery_embedding_mode=info['gallery_embedding_mode'],gallery_mode=mode,
                            return_records=True,table_queries=100,table_top_k=5,
                            batch_size=2,query_batch_size=32,gallery_crop_batch_size=32)
        records=result.pop('retrieval_records')
        retrieval_records[(spec['name'],mode)]=records
        row=dict(system=spec['name'],checkpoint_epoch=info['epoch'],selection=spec['selection'],
                 evaluation_wall_seconds=time.perf_counter()-started,**result)
        rows.append(row)
        # Save after each evaluation, before visualization work.
        pd.DataFrame(rows).to_csv(OUTPUT/'metrics.csv',index=False)
        (OUTPUT/f'retrieval-{index}-{mode}.json').write_text(json.dumps(records,indent=2))
        (OUTPUT/'provenance.json').write_text(json.dumps(provenance,indent=2))
    banks[spec['name']]=occurrence_embeddings(model,PRW_ROOT,sample,DEVICE,
                                             info['detector_image_size'],info['gallery_embedding_mode'])
    torch.save(banks,OUTPUT/'visualization_embeddings.pt')
    del model
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
comparison=pd.DataFrame(rows)
display(comparison)


## Accuracy and detection quality
The timing column is full evaluation wall time (I/O, embedding, ranking and logging), not a controlled inference-latency benchmark. Parameter counts include serialized detector/ReID heads, including unused legacy detector ReID parameters. These should not be presented as latency or active-parameter benchmarks.


In [ ]:
accuracy=comparison.pivot(index='system',columns='gallery_mode',values=['retrieval_mAP','retrieval_top1'])*100
accuracy=accuracy.reindex([s['name'] for s in active])
display(accuracy.round(2))
accuracy.to_csv(OUTPUT/'accuracy_percent.csv')
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,metric,title in zip(axes,['retrieval_mAP','retrieval_top1'],['mAP (%)','Top-1 (%)']):
    accuracy[metric].plot.bar(ax=ax,ylim=(0,100),rot=20)
    ax.set_title(title);ax.set_xlabel('');ax.legend(title='Gallery boxes')
fig.tight_layout();fig.savefig(OUTPUT/'accuracy.png',dpi=160,bbox_inches='tight');plt.show()
detection_columns=[c for c in comparison if c.startswith('detection_')]
display(comparison[comparison.gallery_mode=='detected'][['system']+detection_columns])


In [ ]:
# Re-run visualization cells without repeating model evaluation.
fig=plot_embeddings(banks,sample,seed=SEED)
fig.savefig(OUTPUT/'tsne.png',dpi=180,bbox_inches='tight');plt.show()
fig=plot_similarities(banks,sample)
fig.savefig(OUTPUT/'similarity.png',dpi=180,bbox_inches='tight');plt.show()


## Identical queries, top-five retrievals
Use the first ten query keys common to every system's saved records, in official query order. Queries are not selected by performance. Green/red reflects identity and IoU correctness, including duplicate handling; the same query is shown once per model row. Full-frame box overlays are available in the optional W&B retrieval tables.


In [ ]:
matched={s['name']:retrieval_records[(s['name'],'detected')] for s in active}
key_sets=[{(r['identity'],r['frame']) for r in rs} for rs in matched.values()]
common=set.intersection(*key_sets)
first=next(iter(matched.values()))
query_keys=list(dict.fromkeys((r['identity'],r['frame']) for r in first if (r['identity'],r['frame']) in common))[:NUM_EXAMPLE_QUERIES]
if len(query_keys)<NUM_EXAMPLE_QUERIES:print('Only',len(query_keys),'common queries available')
(OUTPUT/'example_query_keys.json').write_text(json.dumps(query_keys))
for i,key in enumerate(query_keys):
    fig=plot_matched_query(matched,PRW_ROOT,key)
    fig.savefig(OUTPUT/f'query-comparison-{i:02d}.png',dpi=130,bbox_inches='tight')
    plt.show();plt.close(fig)


In [ ]:
# Optional upload of final artifacts/visualizations; never resumes a training run.
if LOG_TO_WANDB:
    import wandb
    if wandb.run is not None:wandb.finish()
    report_run=wandb.init(entity='models-alma-mater-studio',project='person-search',
                          name='final-system-comparison',job_type='evaluation',
                          config=dict(score_threshold=SCORE_THRESHOLD,iou_threshold=IOU_THRESHOLD,
                                      visualization_seed=SEED,systems=provenance))
    try:
        report_run.log({'comparison/results':wandb.Table(dataframe=comparison),
                        'comparison/accuracy':wandb.Image(str(OUTPUT/'accuracy.png')),
                        'comparison/tsne':wandb.Image(str(OUTPUT/'tsne.png')),
                        'comparison/similarity':wandb.Image(str(OUTPUT/'similarity.png')),
                        'comparison/examples':[wandb.Image(str(p)) for p in sorted(OUTPUT.glob('query-comparison-*.png'))]})
        for index,spec in enumerate(active):
            report_run.log({f'comparison/model_{index}_retrievals':make_wandb_retrieval_table(
                retrieval_records[(spec['name'],'detected')],PRW_ROOT)})
    finally:report_run.finish()
print('Submission outputs:',OUTPUT)


## Interpretation and limitations
- Compare complete systems: frozen scene features versus separately fine-tuned crop encoders are not controlled solely by pass count.
- The selected Swin checkpoints are user-specified latest versions; do not call them validation-best unless replaced by verified best artifacts.
- DINO had more training and hyperparameter exploration. Report checkpoint epochs and training settings from provenance.
- Fixed frame-split validation is not identity-disjoint. Test identities are used only for final evaluation/visualization, not training or threshold selection.
- t-SNE and the 20-ID similarity sample illustrate behavior; full-gallery mAP/Top-1 determine retrieval performance.
- ResNet is a historical crop-encoding baseline. A genuine single-pass ResNet comparison requires verified crop-to-scene supervision; changing only its loss label to SupCon does not establish that.
